# STAT 236 · Sparse submatrix detection and recovery
### Reproducible experiments + matrix-size / sparsity-exponent exploration

**Amrit Chadha and Faris Doughan · Fall 2026**

This notebook collects the original paper experiment's **source procedures, correctness checks, archived results, raw trial data, and analyses**, and adds a configurable study over matrix sizes \(n\), sparsity exponents \(\gamma\), and mean shifts \(\mu\). It runs in **Google Colab** and standard Jupyter with Python 3.

**Experiment:** Under \(H_0\), independently sample an \(n\times n\) matrix of \(N(0,1)\) entries. Under \(H_1\), choose \(k\) rows and \(k\) columns uniformly, independently, and add \(\mu>0\) to their \(k^2\) intersections. Row and column sets **need not be consecutive**. Neither detection nor recovery methods receive the true supports.

**Two distinct datasets:**
- **Archived paper runs**: the original 50,400 evaluated matrices/trials and fixed summary, loaded directly from the public project repository. These results are not overwritten.
- **New exploratory sweep**: reproducible new simulation runs on selected \((n,\gamma,\mu)\) settings. Their outputs go into `outputs_gamma_sweep/`. Do **not** cite them as the paper's preregistered or previously validated results without review.

**Statistical caution:** Null calibration, null evaluation, and planted alternatives use independent random draws. Reported 95% Wilson intervals describe finite evaluation-sample uncertainty, *not* uncertainty from calibrating a threshold. Comparisons across methods are paired by matrix; their separate intervals are not CIs for differences in performance. No finite-sample experiment establishes asymptotic minimax limits.

## 1. Configuration
Change the values below and run all cells. `MODE = "demo"` executes quickly; `standard` provides intermediate precision; `study` uses 1,500 calibration draws and 600 evaluation draws per condition. For larger studies, set `ONLY_N` and rerun by matrix size—the notebook saves each configuration as it completes. You can also select `fixed_mu` or `power_law` signal parameterizations.

In [ ]:
# The notebook is a full implementation: no custom project modules need importing.
# Colab ships with NumPy, pandas and Matplotlib; install only if missing.
import os, sys, json, csv, time, math, platform, io
from pathlib import Path
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

GITHUB_REPO = 'avchadha/STAT-236-PROJECT-CHADHA-DOUGHAN'
RANDOM_SEED = 2362026
ALPHA_SWEEP = 0.05
MATRIX_SIZES = [6, 8, 12, 16, 24, 32, 64]
GAMMAS = [0.25, 0.40, 0.55, 0.70, 0.85]

# Pick one. "relative_scan" is most informative across very different n and k.
SIGNAL_MODE = 'relative_scan'    # 'relative_scan', 'fixed_mu', or 'power_law'
SIGNAL_PARAMETERS = [0.0, 0.65, 1.0, 1.35, 1.8]   # multiples of 2*sqrt(log(n/k)/k)
# For SIGNAL_MODE='fixed_mu', use e.g. [0., .5, 1., 1.5, 2.]
# For SIGNAL_MODE='power_law', use beta values, e.g. [0., .10, .20, .30, .40], mu=n**(-beta)
MODE = 'demo'                   # 'demo', 'standard', 'study'
REPETITIONS = {'demo': (250, 75), 'standard': (600, 200), 'study': (1500, 600)}  # (null-calibration, evaluation per condition)
CALIBRATION_REPS, EVAL_REPS = REPETITIONS[MODE]
RECOVERY_REPS = EVAL_REPS
MAX_EXACT_ROW_SUBSETS = 2500    # only run true exhaustive scan if comb(n,k) <= this
RUN_NEW_SWEEP = True
LOAD_ARCHIVED_DATA = True       # Downloads original results from GitHub if absent locally
RERUN_ORIGINAL_PAPER = False    # Takes longer; archive is loaded below without rerunning
# Set ONLY_N (e.g. 16) to run just one matrix size; re-run for each n to build up a long study.
# ONLY_K may further restrict the configurations, e.g. ONLY_N=16, ONLY_K=4.
ONLY_N = None
ONLY_K = None
RESUME_COMPLETED = True
import hashlib
SETTINGS_TAG = hashlib.sha256(json.dumps(
    [RANDOM_SEED, ALPHA_SWEEP, CALIBRATION_REPS, EVAL_REPS,
     SIGNAL_MODE, SIGNAL_PARAMETERS, MAX_EXACT_ROW_SUBSETS],sort_keys=True).encode()).hexdigest()[:10]
OUTPUT = Path('outputs_gamma_sweep') / f'{MODE}_{SIGNAL_MODE}_{SETTINGS_TAG}'
OUTPUT.mkdir(parents=True,exist_ok=True)
assert SIGNAL_MODE in {'relative_scan', 'fixed_mu', 'power_law'}
assert MODE in REPETITIONS and all(0 < g < 1 for g in GAMMAS)
assert min(MATRIX_SIZES) >= 4 and EVAL_REPS > 0 and CALIBRATION_REPS >= 100
print(f'Settings: n={MATRIX_SIZES}; gamma={GAMMAS}; mode={MODE}; seed={RANDOM_SEED}')

## 2. Verified statistical implementation
The code cell below is copied directly from the existing project `experiments/run_experiments.py`, with **only its working-directory line and automatic command-line invocation adjusted for Jupyter**. This preserves the actual methods, calibration choices, and paper scenario configurations.

In [ ]:
#!/usr/bin/env python3
"""Reproducible Gaussian square-submatrix detection and exact-recovery experiments.

Requires numpy, scipy, matplotlib. All paper statistics are generated from
independent, seeded Monte Carlo samples, NOT from website visitor simulations.
"""
from __future__ import annotations

import argparse
import csv
import itertools
import json
import math
import os
import platform
import time
from pathlib import Path

import numpy as np

ROOT = Path.cwd()  # Colab-compatible working directory
RESULTS = ROOT / 'results'
DEFAULT_SEED = 2362026
ALPHA = 0.05

DETECTION = [
    dict(id='D1', n=16, k=4, mus=[0, .4, .8, 1.2, 1.6, 2., 2.4], scan=True),
    dict(id='D2', n=12, k=5, mus=[0, .2, .4, .6, .8, 1., 1.2], scan=True),
    dict(id='D3', n=64, k=16, mus=[0, .15, .3, .45, .6, .75, .9], scan=False),
    dict(id='D4', n=64, k=4, mus=[0, .5, 1., 1.5, 2., 2.5, 3.], scan=False),
]
RECOVERY = [
    dict(id='R1', n=16, k=4, mus=[0, .75, 1., 1.25, 1.5, 1.75, 2., 2.5], exhaustive=True, thresholded=False),
    dict(id='R2', n=12, k=5, mus=[0, .35, .5, .65, .8, 1., 1.2], exhaustive=True, thresholded=False),
    dict(id='R3', n=64, k=16, mus=[0, .4, .6, .8, 1., 1.2, 1.5], exhaustive=False, thresholded=False),
    dict(id='R4', n=64, k=4, mus=[0, 1., 2., 3., 4., 5.], exhaustive=False, thresholded=True),
]


def wilson(successes: int, total: int, z: float = 1.95996398454) -> tuple[float, float]:
    """Two-sided 95% Wilson score interval (including 0 and 1 cases)."""
    p = successes / total
    d = 1 + z*z/total
    center = (p + z*z/(2*total)) / d
    width = z*math.sqrt((p*(1-p)+z*z/(4*total))/total) / d
    return max(0., center-width), min(1., center+width)


def planted_matrix(n: int, k: int, mu: float, rng: np.random.Generator):
    X = rng.standard_normal((n, n))
    S = np.sort(rng.choice(n, k, replace=False))
    T = np.sort(rng.choice(n, k, replace=False))
    X[np.ix_(S, T)] += mu
    return X, S, T


ROW_COMBOS: dict[tuple[int, int], np.ndarray] = {}


def exact_scan(X: np.ndarray, k: int, *, return_indices: bool = False):
    """Exactly maximize sum over ALL k x k row/column index subsets.

    Row subsets are enumerated. For each, maximization over column subsets
    is solved exactly by choosing its k largest aggregated column sums.
    The max is not restricted to the planted location.
    """
    n = X.shape[0]
    key = (n, k)
    if key not in ROW_COMBOS:
        if math.comb(n, k) > 100_000:
            raise ValueError('Exhaustive scan disabled above 100,000 row subsets')
        ROW_COMBOS[key] = np.array(list(itertools.combinations(range(n), k)), dtype=np.int16)
    combos = ROW_COMBOS[key]
    best_value = -np.inf
    best_rows = best_cols = None
    # Batching ensures bounded RAM even for moderately large candidate counts.
    for start in range(0, len(combos), 512):
        rowsets = combos[start:start+512]
        column_sums = X[rowsets].sum(axis=1)
        # Deterministic tie handling; exact continuous Gaussian data ties have probability zero.
        top = np.argsort(column_sums, axis=1)[:, -k:]
        values = np.take_along_axis(column_sums, top, axis=1).sum(axis=1)
        idx = int(np.argmax(values))
        if values[idx] > best_value:
            best_value = float(values[idx])
            best_rows = rowsets[idx].copy()
            best_cols = np.sort(top[idx]).astype(np.int16)
    if return_indices:
        return best_value / k, best_rows, best_cols
    return best_value / k


def test_statistics(X: np.ndarray, k: int, include_scan: bool):
    n = len(X)
    out = {
        'sum': float(X.sum()/n),
        'maximum': float(np.max(X)),
        'column_pool': float(X.sum(axis=0).max()/math.sqrt(n)),
    }
    if include_scan:
        out['scan'] = exact_scan(X, k)
    return out


def empirical_cutoff(values: list[float], alpha: float):
    """Conservative split-calibration order statistic.

    Under the continuous IID null, a fresh null statistic exceeds the
    m-th ascending order statistic with probability (N+1-m)/(N+1).
    """
    N = len(values)
    rank = math.ceil((N+1)*(1-alpha))  # 1-based
    if rank > N:
        raise ValueError('Not enough calibration draws for requested alpha')
    return float(np.sort(values)[rank-1])


def detection_decisions(stats: dict, critical: dict):
    flags = {name: int(val > critical[name]) for name, val in stats.items()}
    if 'scan' in stats:
        flags['sum_or_scan'] = int(stats['sum'] > critical['sum_bonferroni'] or stats['scan'] > critical['scan_bonferroni'])
    return flags


def recovery_spectral(X: np.ndarray, k: int, threshold: float | None = None):
    if threshold is None:
        Y = X
    else:
        Y = np.sign(X) * np.maximum(np.abs(X)-threshold, 0.)
    u, _, vt = np.linalg.svd(Y, full_matrices=False)
    u = u[:, 0].copy()
    v = vt[0, :].copy()
    # The rank-one mean is positive; orient both singular vectors jointly.
    if float(u.sum()) < 0.:
        u *= -1.
        v *= -1.
    row_scores = Y @ v
    col_scores = Y.T @ u
    rows = np.sort(np.argsort(row_scores)[-k:])
    cols = np.sort(np.argsort(col_scores)[-k:])
    return rows, cols


def jaccard(a: np.ndarray, b: np.ndarray):
    match = np.intersect1d(a,b,assume_unique=True).size
    return float(match / (2*len(a)-match))


def export_csv(path: Path, rows: list[dict]):
    if not rows:
        raise ValueError('Refusing to write empty CSV')
    fields = list(dict.fromkeys(k for row in rows for k in row.keys()))
    with path.open('w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        w.writerows(rows)


def run_detection(scenario, rng, cal_reps, reps, summary, trials, cutoffs):
    n, k, scan = scenario['n'], scenario['k'], scenario['scan']
    keys = ['sum','maximum','column_pool'] + (['scan'] if scan else [])
    calibrations = {method: [] for method in keys}
    for c in range(cal_reps):
        X = rng.standard_normal((n,n))
        ss = test_statistics(X,k,scan)
        for name in keys:
            calibrations[name].append(ss[name])
    critical = {name: empirical_cutoff(vals, ALPHA) for name, vals in calibrations.items()}
    if scan:
        critical['sum_bonferroni'] = empirical_cutoff(calibrations['sum'], ALPHA/2)
        critical['scan_bonferroni'] = empirical_cutoff(calibrations['scan'], ALPHA/2)
    for name, val in critical.items():
        cutoffs.append(dict(scenario=scenario['id'], n=n, k=k, statistic=name, calibration_reps=cal_reps,
                            alpha=(ALPHA/2 if 'bonferroni' in name else ALPHA), critical_value=val))
    all_methods = keys + (['sum_or_scan'] if scan else [])
    for mu in scenario['mus']:
        buckets = {name: {'null':0,'signal':0} for name in all_methods}
        runtime = {name: {'null':0.,'signal':0.} for name in keys}
        for condition in ('null','signal'):
            for rep in range(reps):
                if condition == 'null':
                    X = rng.standard_normal((n,n))
                else:
                    X, _, _ = planted_matrix(n,k,mu,rng)
                # All methods see the EXACT same matrix in each replicate.
                stats = {}
                for name in keys:
                    start = time.perf_counter()
                    if name == 'sum': value = float(X.sum()/n)
                    elif name == 'maximum': value = float(X.max())
                    elif name == 'column_pool': value = float(X.sum(axis=0).max()/math.sqrt(n))
                    else: value = exact_scan(X,k)
                    runtime[name][condition] += time.perf_counter() - start
                    stats[name] = value
                flags = detection_decisions(stats, critical)
                for name in all_methods:
                    buckets[name][condition] += flags[name]
                trial = dict(task='detection',scenario=scenario['id'],n=n,k=k,mu=mu,
                             condition=condition,rep=rep)
                for name,value in stats.items(): trial['stat_'+name] = round(value,10)
                for name,flag in flags.items(): trial['reject_'+name] = flag
                trials.append(trial)
        for name in all_methods:
            for condition in ('null','signal'):
                count = buckets[name][condition]
                low, high = wilson(count,reps)
                summary.append(dict(task='detection',scenario=scenario['id'],n=n,k=k,mu=mu,method=name,
                                   metric=('false_positive_rate' if condition=='null' else 'power'),
                                   count=count,replicates=reps,estimate=count/reps,ci_low=low,ci_high=high,
                                   alpha=ALPHA,calibration_reps=cal_reps,
                                   mean_eval_seconds=(runtime[name][condition]/reps if name in runtime else None)))
        print(f"  {scenario['id']} mu={mu:.2f} power: " + ', '.join(f"{x}:{buckets[x]['signal']/reps:.2f}" for x in all_methods), flush=True)


def run_recovery(scenario, rng, reps, summary, trials):
    n, k = scenario['n'], scenario['k']
    methods = (['exhaustive'] if scenario['exhaustive'] else [])+['spectral'] + (['soft_spectral'] if scenario['thresholded'] else [])
    threshold = math.sqrt(2 * math.log(n/(k*k))) if scenario['thresholded'] else None
    for mu in scenario['mus']:
        exact = {name:0 for name in methods}
        overlap = {name:0. for name in methods}
        runtime = {name:0. for name in methods}
        for rep in range(reps):
            X, S, T = planted_matrix(n,k,mu,rng)
            trial = dict(task='recovery',scenario=scenario['id'],n=n,k=k,mu=mu,condition='signal',rep=rep)
            for name in methods:
                start=time.perf_counter()
                if name == 'exhaustive':
                    _, sr, tc = exact_scan(X,k,return_indices=True)
                elif name=='spectral':
                    sr, tc = recovery_spectral(X,k)
                else:
                    sr, tc = recovery_spectral(X,k,threshold=threshold)
                runtime[name] += time.perf_counter()-start
                ok = int(np.array_equal(S,sr) and np.array_equal(T,tc))
                part=(jaccard(S,sr)+jaccard(T,tc))/2
                exact[name]+=ok
                overlap[name]+=part
                trial['exact_'+name]=ok
                trial['jaccard_'+name]=round(part,8)
            trials.append(trial)
        for name in methods:
            lo,hi=wilson(exact[name],reps)
            summary.append(dict(task='recovery',scenario=scenario['id'],n=n,k=k,mu=mu,method=name,
                                metric='exact_recovery',count=exact[name],replicates=reps,estimate=exact[name]/reps,
                                ci_low=lo,ci_high=hi,alpha=None,calibration_reps=None,
                                mean_eval_seconds=runtime[name]/reps))
            summary.append(dict(task='recovery',scenario=scenario['id'],n=n,k=k,mu=mu,method=name,
                                metric='mean_index_jaccard',count=None,replicates=reps,estimate=overlap[name]/reps,
                                ci_low=None,ci_high=None,alpha=None,calibration_reps=None,
                                mean_eval_seconds=runtime[name]/reps))
        print(f"  {scenario['id']} mu={mu:.2f} exact: " + ', '.join(f"{x}:{exact[x]/reps:.2f}" for x in methods), flush=True)


def main():
    parser=argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--mode',choices=['quick','full'],default='full')
    parser.add_argument('--seed',type=int,default=DEFAULT_SEED)
    parser.add_argument('--only',default='',help='Optional scenario ID, e.g. D1 or R2')
    args=parser.parse_args()
    os.environ.setdefault('OPENBLAS_NUM_THREADS','1')
    # Unlike setting these at import time, this may not affect an already-loaded BLAS.
    cal_reps, reps = (250, 100) if args.mode == 'quick' else (1500,600)
    if args.only and args.only not in [s['id'] for s in DETECTION+RECOVERY]:
        parser.error('Unknown scenario ID')
    RESULTS.mkdir(parents=True,exist_ok=True)
    # Separate independent deterministic streams by scenario, so --only D2
    # reproduces the D2 block of a full run byte-for-byte (timings aside).
    def stream(scenario_id):
        return np.random.default_rng(np.random.SeedSequence(
            [args.seed, ord(scenario_id[0]), int(scenario_id[1:])]))
    summary, trials, cutoffs = [],[],[]
    start=time.perf_counter()
    for scenario in DETECTION:
        if args.only and scenario['id']!=args.only: continue
        print('Starting',scenario['id'],'detection',flush=True)
        run_detection(scenario,stream(scenario['id']),cal_reps,reps,summary,trials,cutoffs)
    for scenario in RECOVERY:
        if args.only and scenario['id']!=args.only: continue
        print('Starting',scenario['id'],'recovery',flush=True)
        run_recovery(scenario,stream(scenario['id']),reps,summary,trials)
    name = '' if not args.only else '_'+args.only
    export_csv(RESULTS/f'summary{name}.csv',summary)
    export_csv(RESULTS/f'trials{name}.csv',trials)
    if cutoffs: export_csv(RESULTS/f'calibration{name}.csv',cutoffs)
    meta = dict(seed=args.seed, alpha=ALPHA, mode=args.mode, calibration_reps=cal_reps,
                null_validation_reps_per_signal=reps, alternative_reps_per_signal=reps,
                recovery_reps_per_signal=reps, time_seconds=time.perf_counter()-start,
                scenario_rng='SeedSequence([master_seed, ord(task_letter), scenario_index])',
                numpy_version=np.__version__, platform=platform.platform(), python=platform.python_version(),
                detection_scenarios=DETECTION, recovery_scenarios=RECOVERY,
                note=('The same held-out null matrices are redrawn independently for every mu; '+
                      'the same planted alternative is shared across procedures within a replicate. '+
                      'The exhaustive scan is exact, with arbitrary nonconsecutive supports.'))
    (RESULTS/f'metadata{name}.json').write_text(json.dumps(meta,indent=2)+'\n')
    print('Finished in',round(meta['time_seconds'],2),'seconds; wrote',len(summary),'summary rows and',len(trials),'trial rows.',flush=True)

# Invoke main() only from the optional paper-rerun cell below.


## 3. Correctness checks
Before using the results, verify that the optimized exhaustive scan matches brute force on small matrices; check easy-case support recovery, the empirical calibration order statistic, and Wilson interval endpoints.

In [ ]:
def validate_implementations():
    rng=np.random.default_rng(42)
    for n,k in ((5,2),(6,3)):
        for _ in range(6):
            X=rng.standard_normal((n,n))
            got, rows, cols=exact_scan(X,k,return_indices=True)
            brute=max(X[np.ix_(a,b)].sum()/k for a in itertools.combinations(range(n),k)
                      for b in itertools.combinations(range(n),k))
            assert np.isclose(got,brute,atol=1e-12)
            assert np.isclose(X[np.ix_(rows,cols)].sum()/k,brute,atol=1e-12)
    rng=np.random.default_rng(188)
    for n,k in ((16,4),(64,16)):
        X,S,T=planted_matrix(n,k,20.,rng)
        A,B=recovery_spectral(X,k)
        assert np.array_equal(S,A) and np.array_equal(T,B)
    assert empirical_cutoff(list(range(99)),.05)==94
    low,high=wilson(0,400)
    assert low==0 and 0<high<.02
    assert math.isclose(wilson(100,200)[0], 1-wilson(100,200)[1], abs_tol=1e-10)
    print('PASS — scan, maximizing supports, strong-signal recovery, calibration and intervals')
validate_implementations()

## 4. Load and audit the original archived paper data
This section retrieves the **actual immutable paper-run records**, not fabricated or newly simulated substitutes. Colab downloads them from the GitHub repository; standard Jupyter can use a local checked-out `results/` folder. The data are not embedded into the notebook because that would substantially inflate the `.ipynb` file.

In [ ]:
from urllib.request import urlopen
BASE_RAW = f'https://raw.githubusercontent.com/{GITHUB_REPO}/main/'
def csv_from_repo(path):
    local=Path(path)
    if local.is_file():
        return pd.read_csv(local)
    return pd.read_csv(BASE_RAW+path)

def json_from_repo(path):
    local=Path(path)
    if local.is_file():
        return json.loads(local.read_text())
    with urlopen(BASE_RAW+path,timeout=30) as r:
        return json.load(r)

paper_summary=pd.DataFrame(); paper_trials=pd.DataFrame(); paper_calibration=pd.DataFrame()
if LOAD_ARCHIVED_DATA:
    try:
        paper_summary=csv_from_repo('results/summary.csv')
        paper_calibration=csv_from_repo('results/calibration.csv')
        paper_metadata=json_from_repo('results/metadata.json')
        parts=[csv_from_repo(f'results/trial_parts/trials_{j:02d}.csv') for j in range(1,11)]
        paper_trials=pd.concat(parts, ignore_index=True)
        assert len(paper_trials)==50400, f'Unexpected paper record count: {len(paper_trials)}'
        assert len(paper_summary)==322, f'Unexpected paper summary count: {len(paper_summary)}'
        assert paper_metadata['seed']==2362026
        assert set(paper_trials.task)=={'detection','recovery'}
        print('Loaded archived paper data:',len(paper_trials),'trials,',len(paper_summary),'summary entries')
        display(paper_summary.head(8))
    except Exception as ex:
        print('Archived data could not be loaded:',type(ex).__name__,str(ex))
        print('New simulations below still run; check internet access or supply local results/ files.')

### Original paper comparisons
These plots come **only from the archived CSV**, not the new gamma sweep. They provide the reference points that appear in the website and current simulation report.

In [ ]:
METHOD_LABELS={'sum':'Global sum','scan':'Exact scan','maximum':'Max entry',
 'column_pool':'Column pooling','sum_or_scan':'Sum or scan',
 'exhaustive':'Maximizing block','spectral':'Spectral','soft_spectral':'Thresholded spectral'}
METHOD_COLORS={'sum':'#a51c30','scan':'#20232a','maximum':'#7c858d',
 'column_pool':'#b28991','sum_or_scan':'#75313e',
 'exhaustive':'#a51c30','spectral':'#303943','soft_spectral':'#9a6b75'}

def plot_archived(metric,scenario_ids):
    if paper_summary.empty:
        print('Archive unavailable; skipping.'); return
    fig,axs=plt.subplots(1,len(scenario_ids),figsize=(5*len(scenario_ids),3.9),squeeze=False)
    for ax,sid in zip(axs[0],scenario_ids):
        table=paper_summary.query('scenario==@sid and metric==@metric')
        for name,gg in table.groupby('method'):
            gg=gg.sort_values('mu')
            color=METHOD_COLORS.get(name,'black')
            ax.plot(gg.mu,gg.estimate,'o-',label=METHOD_LABELS.get(name,name),color=color,linewidth=1.6,markersize=3)
            ax.fill_between(gg.mu.to_numpy(),gg.ci_low.to_numpy(),gg.ci_high.to_numpy(),color=color,alpha=.10)
        ax.set(title=f'{sid} · n={int(table.n.iloc[0])}, k={int(table.k.iloc[0])}',xlabel='Mean shift μ',ylim=(-.03,1.03))
        ax.grid(axis='y',alpha=.15)
        ax.spines[['top','right']].set_visible(False)
    axs[0][0].set_ylabel(metric.replace('_',' ').capitalize())
    handles,labels=axs[0][0].get_legend_handles_labels()
    fig.legend(handles,labels,loc='lower center',bbox_to_anchor=(.5,-.03),ncol=min(5,len(labels)),frameon=False)
    fig.tight_layout(rect=(0,.08,1,1))
    fig.savefig(OUTPUT/f'archived_{metric}.png',dpi=180,bbox_inches='tight')
    plt.show()
if not paper_summary.empty:
    plot_archived('power',['D1','D3','D4'])
    plot_archived('exact_recovery',['R1','R3','R4'])

## 5. New study: vary \(n\), \(\gamma\), and \(\mu\)
We set \(k=\max\{2,\min(n-1,\lfloor n^\gamma\rfloor)\}\). The clipping at 2 is a finite-size convenience, **not part of the asymptotic model**. We also record \(\gamma_{\mathrm{eff}}=\log(k)/\log(n)\) to show when rounding matters. If two requested \(\gamma\) values give the same \((n,k)\), we simulate once and reuse the result; this avoids mistaking duplicate block sizes for separate evidence.

Three signal grids are supported:
- **`relative_scan` (recommended):** \(\mu=c\,2\sqrt{\log(n/k)/k}\), with multiplier \(c\) in `SIGNAL_PARAMETERS`. This tracks a natural scale from the scan test's detection condition. It is a finite-sample *reference scale*, not an exact optimal boundary for all methods.
- **`fixed_mu`:** \(\mu=c\) for each chosen mean shift \(c\).
- **`power_law`:** \(\mu=n^{-\beta}\) for each chosen \(\beta\). Useful to compare to phase-diagram exponents, but at small \(n\) the asymptotics should not be overinterpreted.

For every distinct \((n,k)\), calibrate each detector on independent null matrices at level 0.05 (combined test uses 0.025 cutoffs), then evaluate **new null** and **new planted** matrices at each signal value. Recovery uses independent planted matrices and requires both supports to match exactly. Exact scan and exhaustive recovery run only if \(\binom{n}{k}\leq\) `MAX_EXACT_ROW_SUBSETS`. This bound applies to row subsets, not to a heuristic approximation of the scan statistic.

In [ ]:
def k_from_gamma(n,gamma):
    return max(2,min(n-1,int(math.floor(n**gamma))))

def signal_mu(n,k,parameter):
    if SIGNAL_MODE=='relative_scan':
        return float(parameter * 2 * math.sqrt(math.log(n/k)/k))
    if SIGNAL_MODE=='fixed_mu':
        return float(parameter)
    return float(n**(-parameter))

mapping=[]
for n in MATRIX_SIZES:
    for target in GAMMAS:
        k=k_from_gamma(n,target)
        mapping.append(dict(n=n, gamma_requested=target, k=k,
                            gamma_effective=math.log(k)/math.log(n), row_subsets=math.comb(n,k)))
gamma_mapping=pd.DataFrame(mapping)
unique_configs=(gamma_mapping.sort_values(['n','k'])
               .drop_duplicates(['n','k'])[['n','k','gamma_effective','row_subsets']]
               .to_dict('records'))
print(f'{len(mapping)} requested grid positions => {len(unique_configs)} distinct (n,k) configurations')
display(gamma_mapping.pivot(index='n',columns='gamma_requested',values='k'))
gamma_mapping.to_csv(OUTPUT/'gamma_mapping.csv',index=False)

### Run the exploratory sweep
The code reuses the exact tested functions above and saves **every trial**, **every calibration cutoff**, and **every summary row**. Results are checkpointed after each distinct matrix configuration. If a long study is interrupted, rerun with the same settings to resume; or set `ONLY_N` and build the study one matrix size at a time. The random stream for each pair depends only on the seed and that pair, not the run order.

In [ ]:
from contextlib import redirect_stdout

def run_gamma_sweep():
    batchdir=OUTPUT/'batches';batchdir.mkdir(parents=True,exist_ok=True)
    t0=time.perf_counter()
    eligible_configs=[c for c in unique_configs
                      if (ONLY_N is None or int(c['n'])==int(ONLY_N))
                      and (ONLY_K is None or int(c['k'])==int(ONLY_K))]
    if not eligible_configs:
        raise ValueError('No distinct (n,k) configurations match ONLY_N and ONLY_K.')
    for idx,cfg in enumerate(eligible_configs,1):
        n,k=int(cfg['n']),int(cfg['k'])
        prefix=batchdir/f'n{n}_k{k}'
        existing=[Path(str(prefix)+'_'+name+'.csv') for name in ('summary','trials','calibration')]
        if RESUME_COMPLETED and all(f.exists() for f in existing):
            print(f'[{idx:02d}/{len(eligible_configs)}] n={n:>2}, k={k:>2}: checkpoint present',flush=True)
            continue
        eligible=math.comb(n,k)<=MAX_EXACT_ROW_SUBSETS
        mus=[signal_mu(n,k,x) for x in SIGNAL_PARAMETERS]
        mus=list(dict.fromkeys(round(float(x),12) for x in mus))
        gamma_eff=math.log(k)/math.log(n)
        d=dict(id=f'D_{n}_{k}',n=n,k=k,mus=mus,scan=eligible)
        rr=dict(id=f'R_{n}_{k}',n=n,k=k,mus=mus,exhaustive=eligible,
                thresholded=(n>k*k))
        sb=[];tb=[];cb=[]
        rng_d=np.random.default_rng(np.random.SeedSequence([RANDOM_SEED,101,n,k]))
        rng_r=np.random.default_rng(np.random.SeedSequence([RANDOM_SEED,202,n,k]))
        with redirect_stdout(io.StringIO()):
            run_detection(d,rng_d,CALIBRATION_REPS,EVAL_REPS,sb,tb,cb)
            run_recovery(rr,rng_r,RECOVERY_REPS,sb,tb)
        for row in sb:
            row['gamma_effective']=gamma_eff
            row['scan_eligible']=eligible
            row['signal_mode']=SIGNAL_MODE
            row['signal_parameter']=min(SIGNAL_PARAMETERS,key=lambda p:abs(signal_mu(n,k,p)-row['mu']))
        for row in tb:
            row['gamma_effective']=gamma_eff
            row['signal_mode']=SIGNAL_MODE
        for row in cb:
            row['gamma_effective']=gamma_eff
        pd.DataFrame(sb).to_csv(existing[0],index=False)
        pd.DataFrame(tb).to_csv(existing[1],index=False)
        pd.DataFrame(cb).to_csv(existing[2],index=False)
        print(f'[{idx:02d}/{len(eligible_configs)}] n={n:>2}, k={k:>2}, '
              f'gamma_eff={gamma_eff:.3f}, exact_scan={eligible}, saved {len(tb)} trials',flush=True)
    # Read all completed checkpoints so a resumed run includes previous matrix sizes.
    def load_all(name):
        matches=sorted(batchdir.glob(f'n*_k*_{name}.csv'))
        if not matches: raise ValueError('No batches available: '+name)
        return pd.concat([pd.read_csv(q) for q in matches],ignore_index=True)
    df,trials,cut=load_all('summary'),load_all('trials'),load_all('calibration')
    df.to_csv(OUTPUT/'gamma_sweep_summary.csv',index=False)
    trials.to_csv(OUTPUT/'gamma_sweep_trials.csv',index=False)
    cut.to_csv(OUTPUT/'gamma_sweep_calibration.csv',index=False)
    completed_pairs=set(zip(df.n.astype(int),df.k.astype(int)))
    meta=dict(seed=RANDOM_SEED,alpha=ALPHA,calibration_reps=CALIBRATION_REPS,
              evaluation_reps=EVAL_REPS,recovery_reps=RECOVERY_REPS,mode=MODE,
              matrix_sizes=MATRIX_SIZES,gammas=GAMMAS,signal_mode=SIGNAL_MODE,
              signal_parameters=SIGNAL_PARAMETERS,exact_row_cap=MAX_EXACT_ROW_SUBSETS,
              distinct_configurations=len(unique_configs),completed_configurations=len(completed_pairs),
              complete=(len(completed_pairs)==len(unique_configs)),trials=len(trials),
              elapsed_seconds_current_session=round(time.perf_counter()-t0,3),
              numpy=np.__version__,python=platform.python_version(),
              note='Independent calibration/evaluation, paired methods within matrices; batch-wise restart safe.')
    (OUTPUT/'gamma_sweep_metadata.json').write_text(json.dumps(meta,indent=2)+'\n')
    print(f'Finished this session: {len(completed_pairs)}/{len(unique_configs)} configurations, '
          f'{len(trials):,} archived exploratory trials, {len(df)} summary rows.')
    return df,trials,cut,meta

if RUN_NEW_SWEEP:
    sweep_summary,sweep_trials,sweep_calibration,sweep_metadata=run_gamma_sweep()
else:
    sweep_summary=pd.read_csv(OUTPUT/'gamma_sweep_summary.csv')
    sweep_trials=pd.read_csv(OUTPUT/'gamma_sweep_trials.csv')
    sweep_calibration=pd.read_csv(OUTPUT/'gamma_sweep_calibration.csv')
    sweep_metadata=json.loads((OUTPUT/'gamma_sweep_metadata.json').read_text())
assert sweep_summary.estimate.between(0,1).all()
assert set(sweep_summary.task)=={'detection','recovery'}
assert len(sweep_calibration)>0

## 6. Analysis: finite-sample power, error, and exact recovery
First compare observed false-positive rates with the nominal 5% target; then inspect the signal-strength response across block sizes and actual \(\gamma_{\mathrm{eff}}\). White heatmap cells indicate an intentionally **uncomputed** exhaustive statistic, not a failure or a zero success rate.

In [ ]:
# Held-out empirical Type I error: one row per (n,k,signal,method).
fpr=sweep_summary.query("metric == 'false_positive_rate'")
check=(fpr.groupby('method').agg(configurations=('estimate','size'),mean_fpr=('estimate','mean'),
                                 max_fpr=('estimate','max')).round(3))
display(check)
fig,ax=plt.subplots(figsize=(8,3.2))
for method in sorted(fpr.method.unique()):
    ss=fpr.query('method==@method')
    ax.scatter([method]*len(ss),ss.estimate,alpha=.35,s=10,color=METHOD_COLORS.get(method,'#666666'))
ax.axhline(ALPHA_SWEEP,color='#a51c30',linewidth=1.7,linestyle='--',label='Nominal 5%')
ax.set(ylabel='Observed false-positive fraction',ylim=(0,max(.18, float(fpr.estimate.max())+.025)))
ax.tick_params(axis='x',rotation=25);ax.grid(axis='y',alpha=.16)
ax.spines[['top','right']].set_visible(False)
fig.tight_layout();fig.savefig(OUTPUT/'held_out_false_positive.png',dpi=180);plt.show()

In [ ]:
# One power / exact-recovery heatmap at a chosen signal-grid point.
chosen_parameter=SIGNAL_PARAMETERS[len(SIGNAL_PARAMETERS)//2]

def heatmap_for(method,metric,ax):
    ss=sweep_summary.query('method==@method and metric==@metric').copy()
    ss=ss[np.isclose(ss.signal_parameter.astype(float),float(chosen_parameter))]
    lookup={(int(r.n),int(r.k)):float(r.estimate) for r in ss.itertuples()}
    mat=np.full((len(MATRIX_SIZES),len(GAMMAS)),np.nan)
    for ix,n in enumerate(MATRIX_SIZES):
        for j,gamma in enumerate(GAMMAS):
            mat[ix,j]=lookup.get((n,k_from_gamma(n,gamma)),float('nan'))
    colors=plt.get_cmap('Reds').copy(); colors.set_bad('#f2f2f2')
    heat=ax.imshow(np.ma.masked_invalid(mat),vmin=0,vmax=1,aspect='auto',cmap=colors)
    ax.set_xticks(range(len(GAMMAS)),labels=[f'{g:g}' for g in GAMMAS])
    ax.set_yticks(range(len(MATRIX_SIZES)),labels=[str(n) for n in MATRIX_SIZES])
    for i,n in enumerate(MATRIX_SIZES):
        for j,g in enumerate(GAMMAS):
            v=mat[i,j]
            ax.text(j,i,'—' if np.isnan(v) else f'{v:.2f}',ha='center',va='center',
                    fontsize=8,color='white' if v>=.65 else '#232323')
    ax.set(xlabel='Requested gamma',ylabel='Matrix size n',title=METHOD_LABELS.get(method,method))
    return heat

fig,axs=plt.subplots(1,3,figsize=(14,5),sharey=True)
for ax,method in zip(axs,['sum','maximum','scan']): heatmap_for(method,'power',ax)
fig.suptitle(f'Detection power · signal parameter = {chosen_parameter:g} ({SIGNAL_MODE})',fontsize=13)
fig.tight_layout(rect=(0,0,1,.95));fig.savefig(OUTPUT/'gamma_n_detection_heatmaps.png',dpi=180);plt.show()

fig,axs=plt.subplots(1,3,figsize=(14,5),sharey=True)
for ax,method in zip(axs,['exhaustive','spectral','soft_spectral']): heatmap_for(method,'exact_recovery',ax)
fig.suptitle(f'Exact recovery · signal parameter = {chosen_parameter:g} ({SIGNAL_MODE})',fontsize=13)
fig.tight_layout(rect=(0,0,1,.95));fig.savefig(OUTPUT/'gamma_n_recovery_heatmaps.png',dpi=180);plt.show()

In [ ]:
# Detailed power curves for selected SMALL configurations (first 3 distinct configurations).
small_configs=[(int(x['n']),int(x['k'])) for x in unique_configs if x['n']<=12][:4]
fig,axs=plt.subplots(1,len(small_configs),figsize=(4.6*len(small_configs),3.8),squeeze=False)
for ax,(n,k) in zip(axs[0],small_configs):
    ss=sweep_summary.query("n==@n and k==@k and metric=='power'")
    for method,group in ss.groupby('method'):
        group=group.sort_values('mu')
        ax.plot(group.mu,group.estimate,marker='o',markersize=3,
                linewidth=1.6,color=METHOD_COLORS.get(method,'#555555'),label=METHOD_LABELS.get(method,method))
        ax.fill_between(group.mu.to_numpy(),group.ci_low.to_numpy(),group.ci_high.to_numpy(),
                        color=METHOD_COLORS.get(method,'#555555'),alpha=.08)
    ax.set(title=f'n={n}, k={k}',xlabel='μ',ylim=(-.03,1.03));ax.grid(axis='y',alpha=.15)
    ax.spines[['top','right']].set_visible(False)
axs[0][0].set_ylabel('Detection power')
handles,labels=axs[0][0].get_legend_handles_labels()
fig.legend(handles,labels,loc='lower center',bbox_to_anchor=(.5,-.02),ncol=min(5,len(labels)),frameon=False)
fig.tight_layout(rect=(0,.1,1,1));fig.savefig(OUTPUT/'small_matrix_power.png',dpi=180,bbox_inches='tight');plt.show()

### Runtime, uncertainty and useful reporting tables
Measured runtime depends on machine, BLAS threading, and implementation. The reported exhaustive scan computes the true maximum, but stops at the declared row-subset budget. For numerical interpretation, compare finite-sample point estimates alongside their Wilson intervals; do not treat tiny differences as established superiority.

In [ ]:
# Representative slice: all procedures at the median signal setting.
select=sweep_summary[np.isclose(sweep_summary.signal_parameter.astype(float),float(chosen_parameter))]
report=select[select.metric.isin(['power','exact_recovery'])]
report=report.sort_values(['task','n','k','method'])
report=report.assign(CI=report.apply(
    lambda r:f"{r.estimate:.3f} [{r.ci_low:.3f}, {r.ci_high:.3f}]",axis=1))
display(report[['task','n','k','gamma_effective','mu','method','metric','CI','mean_eval_seconds']].head(35))
report.to_csv(OUTPUT/'selected_report_table.csv',index=False)

runtime=(report.groupby(['task','method'],as_index=False).mean_eval_seconds.median())
fig,ax=plt.subplots(figsize=(8,3.8))
for task,rr in runtime.groupby('task'):
    ax.scatter(rr.method,rr.mean_eval_seconds,label=task,s=58)
ax.set(yscale='log',ylabel='Median measured time per evaluation (s)')
ax.tick_params(axis='x',rotation=35)
ax.grid(axis='y',alpha=.17);ax.legend(frameon=False)
ax.spines[['top','right']].set_visible(False)
fig.tight_layout();fig.savefig(OUTPUT/'runtime_comparison.png',dpi=180);plt.show()

## 7. Compare with the asymptotic detection phase diagram
For the theoretical power-law scaling \(k\asymp n^\gamma\), \(\mu=n^{-\beta}\), the *statistical detection* exponent boundary is \(\beta_{\mathrm{stat}}=\max\{\gamma/2,2\gamma-1\}\). The **computational boundary** \(\max\{0,2\gamma-1\}\) is conditional on a planted-clique hardness hypothesis and applies to the larger alternative class and discretization studied by Ma and Wu (2015); it must not be stated as an unconditional theorem for our exact-size common-mean simulation model. The curves are asymptotic, not a finite-\(n\) power prediction.

In [ ]:
gamma_axis=np.linspace(.01,.99,300)
beta_stat=np.maximum(gamma_axis/2,2*gamma_axis-1)
beta_comp=np.maximum(0,2*gamma_axis-1)
fig,ax=plt.subplots(figsize=(8.0,4.7))
ax.plot(gamma_axis,beta_stat,color='#a51c30',linewidth=2.1,label='Statistical detection exponent')
ax.plot(gamma_axis,beta_comp,color='#33383e',linewidth=2,linestyle='--',label='Conditional computational exponent')
ax.fill_between(gamma_axis,beta_comp,beta_stat,color='#a51c30',alpha=.09)
ax.set(xlabel='gamma (block-size exponent)',ylabel='beta (signal-strength exponent)',
       title='Asymptotic detection boundaries (for context)',xlim=(0,1),ylim=(0,.9))
ax.legend(frameon=False);ax.grid(alpha=.12);ax.spines[['top','right']].set_visible(False)
fig.tight_layout();fig.savefig(OUTPUT/'theory_phase_diagram.png',dpi=180);plt.show()

## 8. Optional rerun of the original paper scenarios
This cell is disabled by default because the previously archived paper data are already loaded from GitHub. If enabled, it regenerates the **original D1–D4 and R1–R4 designs**, with their original scenario-specific seeded RNG streams, and writes into a separate `outputs_gamma_sweep/paper_rerun/` folder. It never overwrites the archived paper outputs.

In [ ]:
def rerun_original_paper(calibration_reps=1500,eval_reps=600):
    results=[]; trials=[]; cuts=[]
    for s in DETECTION:
        rng=np.random.default_rng(np.random.SeedSequence([RANDOM_SEED,ord(s['id'][0]),int(s['id'][1:])]))
        with redirect_stdout(io.StringIO()):
            run_detection(s,rng,calibration_reps,eval_reps,results,trials,cuts)
        print('Completed original detection',s['id'])
    for s in RECOVERY:
        rng=np.random.default_rng(np.random.SeedSequence([RANDOM_SEED,ord(s['id'][0]),int(s['id'][1:])]))
        with redirect_stdout(io.StringIO()):
            run_recovery(s,rng,eval_reps,results,trials)
        print('Completed original recovery',s['id'])
    p=OUTPUT/'paper_rerun';p.mkdir(exist_ok=True)
    pd.DataFrame(results).to_csv(p/'summary.csv',index=False)
    pd.DataFrame(trials).to_csv(p/'trials.csv',index=False)
    pd.DataFrame(cuts).to_csv(p/'calibration.csv',index=False)
    return pd.DataFrame(results)

if RERUN_ORIGINAL_PAPER:
    paper_recomputed=rerun_original_paper()
    if not paper_summary.empty:
        keys=['task','scenario','n','k','mu','method','metric']
        a=paper_recomputed.merge(paper_summary,on=keys,suffixes=('_new','_archive'),validate='one_to_one')
        compare=a[(a.metric!='mean_index_jaccard')]
        assert (compare.count_new==compare.count_archive).all(), 'Difference in seeded outcome counts!'
        print('Original outcome counts reproduced exactly across all methods/scenarios.')

## 9. Export new results and Colab artifacts
CSV files, raw per-trial records, calibration thresholds, the actual exponent mapping, metadata, a reporting table, and all plotted images are generated in `outputs_gamma_sweep/`. In Colab, the final cell can ZIP the outputs for download. The *source of truth* for the prior paper numbers remains the original GitHub `results/` folder.

In [ ]:
import zipfile
bundle=Path('STAT236_gamma_sweep_outputs.zip')
with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED,compresslevel=7) as archive:
    for path in OUTPUT.rglob('*'):
        if path.is_file():
            archive.write(path,path.as_posix())
print('Saved',bundle.resolve(),f'({bundle.stat().st_size/1024:.1f} KiB)')
print('Files:',len(list(OUTPUT.rglob('*'))))
# In Google Colab, you may uncomment:
# from google.colab import files
# files.download(str(bundle))

## Reproduction checklist

1. Use a single seed and specify the requested \(n\), \(\gamma\), and signal grid.
2. Record the actual \(k\) (and \(\gamma_{\mathrm{eff}}\)) because small matrices make exponent rounding substantial.
3. Calibrate rejection cutoffs **once on independent null draws** at each distinct \((n,k)\).
4. Evaluate fresh null and planted samples; reuse a matrix across competing methods, not across independent calibration and evaluation.
5. Treat scan as *exact* only when all row subsets are enumerated and the best columns are chosen for each.
6. Evaluate recovery against both planted index sets, not merely detection success.
7. Save all trials, thresholds, summary statistics, confidence intervals, and seed metadata before writing conclusions.
8. Keep exploratory findings distinct from the fixed paper results and avoid asserting theory from finite simulated matrices.

**Source literature:** Butucea & Ingster (2013); Ma & Wu (2015); Butucea, Ingster & Suslina (2015); Cai, Liang & Rakhlin (2017). The GitHub repository contains the full report and source references.